# Gradient Covariance PCA Visualization

Simple experiment: Can we distinguish real from fake images using gradient covariance?

### Method
1. **RGB → Luminance** (BT.709)
2. **Compute spatial gradients** (Sobel Gx, Gy)
3. **Flatten gradients** into a 2D matrix
4. **PCA on gradient covariance** to get principal components
5. **Project samples** onto first 2 PCs for visualization

The hypothesis: Real and AI-generated images have different gradient covariance structures.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_DATA_DIR = "/content/drive/MyDrive/datasets"

ModuleNotFoundError: No module named 'google'

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import scipy.ndimage as ndimage

# Data path
data_dir = f"{GDRIVE_DATA_DIR}/OpenFake"

# BT.709 luminance coefficients
R_COEFF, G_COEFF, B_COEFF = 0.2126, 0.7152, 0.0722

---
## 1. Core Functions

In [ ]:
def rgb_to_luminance(img_array):
    """
    Convert RGB image to luminance using BT.709.
    L = 0.2126*R + 0.7152*G + 0.0722*B
    """
    return (R_COEFF * img_array[:,:,0] + 
            G_COEFF * img_array[:,:,1] + 
            B_COEFF * img_array[:,:,2])


def compute_gradients(luminance):
    """
    Compute spatial gradients using Sobel operators.
    Returns Gx (horizontal) and Gy (vertical) gradients.
    """
    Gx = ndimage.sobel(luminance, axis=1)  # Horizontal gradient
    Gy = ndimage.sobel(luminance, axis=0)  # Vertical gradient
    return Gx, Gy


def extract_gradient_covariance_features(img_path, img_size=256):
    """
    Extract gradient covariance features from an image.
    
    Steps:
    1. Load and resize image
    2. Convert RGB to luminance
    3. Compute Sobel gradients Gx, Gy
    4. Flatten gradients and compute covariance features
    
    Returns:
        Feature vector based on gradient covariance
    """
    # Load and resize
    img = Image.open(img_path).convert('RGB')
    img = img.resize((img_size, img_size))
    img_array = np.array(img) / 255.0  # Normalize to [0, 1]
    
    # RGB -> Luminance
    luminance = rgb_to_luminance(img_array)
    
    # Compute gradients
    Gx, Gy = compute_gradients(luminance)
    
    # Flatten gradients into matrix: each row is a pixel, columns are [Gx, Gy]
    # Shape: (H*W, 2)
    gradient_matrix = np.stack([Gx.flatten(), Gy.flatten()], axis=1)
    
    # Compute covariance matrix of gradients (2x2)
    cov_matrix = np.cov(gradient_matrix.T)
    
    # Extract features from covariance:
    # 1. Eigenvalues (λ1, λ2) capture gradient energy distribution
    # 2. Trace = λ1 + λ2 (total gradient energy)
    # 3. Determinant = λ1 * λ2 
    # 4. Anisotropy = (λ1 - λ2) / (λ1 + λ2) (how directional the gradients are)
    
    eigenvalues = np.linalg.eigvalsh(cov_matrix)
    eigenvalues = np.sort(eigenvalues)[::-1]  # Descending order
    
    lambda1, lambda2 = eigenvalues
    trace = lambda1 + lambda2
    det = lambda1 * lambda2
    anisotropy = (lambda1 - lambda2) / (trace + 1e-8)
    
    # Also include raw gradient statistics
    magnitude = np.sqrt(Gx**2 + Gy**2)
    
    features = np.array([
        lambda1,                    # Largest eigenvalue
        lambda2,                    # Smallest eigenvalue
        trace,                      # Total gradient energy
        det,                        # Determinant
        anisotropy,                 # Gradient directionality
        cov_matrix[0, 1],           # Gx-Gy covariance
        magnitude.mean(),           # Mean gradient magnitude
        magnitude.std(),            # Std gradient magnitude
        np.abs(Gx).mean(),          # Mean |Gx|
        np.abs(Gy).mean(),          # Mean |Gy|
    ])
    
    return features

---
## 2. Load Data and Extract Features

In [ ]:
def get_image_paths_and_labels(folder):
    """Get image paths and labels (0=real, 1=fake)."""
    paths, labels = [], []
    for label, subfolder in enumerate(["real", "fake"]):
        subdir = os.path.join(folder, subfolder)
        for fname in os.listdir(subdir):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(subdir, fname))
                labels.append(label)
    return paths, labels

# Load test set (smaller, faster for visualization)
test_paths, test_labels = get_image_paths_and_labels(f"{data_dir}/test")

print(f"Test samples: {len(test_paths)}")
print(f"  Real: {test_labels.count(0)}")
print(f"  Fake: {test_labels.count(1)}")

In [ ]:
# Extract features for all test images
print("Extracting gradient covariance features...")

features = []
for path in tqdm(test_paths):
    feat = extract_gradient_covariance_features(path)
    features.append(feat)

X = np.array(features)
y = np.array(test_labels)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Features: [λ1, λ2, trace, det, anisotropy, cov_xy, mag_mean, mag_std, Gx_mean, Gy_mean]")

---
## 3. PCA Projection

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

---
## 4. Visualization

In [ ]:
# Main visualization: 2D PCA projection
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
real_mask = y == 0
fake_mask = y == 1

axes[0].scatter(X_pca[real_mask, 0], X_pca[real_mask, 1], 
               c='green', alpha=0.6, label='Real', s=50, edgecolors='white', linewidth=0.5)
axes[0].scatter(X_pca[fake_mask, 0], X_pca[fake_mask, 1], 
               c='red', alpha=0.6, label='Fake', s=50, edgecolors='white', linewidth=0.5)

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('Gradient Covariance PCA Projection\nReal vs Fake')
axes[0].legend(loc='upper right', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Add convex hulls
from scipy.spatial import ConvexHull
for mask, color in [(real_mask, 'green'), (fake_mask, 'red')]:
    points = X_pca[mask]
    if len(points) >= 3:
        hull = ConvexHull(points)
        for simplex in hull.simplices:
            axes[0].plot(points[simplex, 0], points[simplex, 1], color=color, alpha=0.3)

# 1D projection histogram (PC1 only)
axes[1].hist(X_pca[real_mask, 0], bins=30, alpha=0.6, color='green', label='Real', density=True)
axes[1].hist(X_pca[fake_mask, 0], bins=30, alpha=0.6, color='red', label='Fake', density=True)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel('Density')
axes[1].set_title('PC1 Distribution\n(Single Projection Separation)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute separation metric
real_mean = X_pca[real_mask, 0].mean()
fake_mean = X_pca[fake_mask, 0].mean()
real_std = X_pca[real_mask, 0].std()
fake_std = X_pca[fake_mask, 0].std()

# Cohen's d (effect size)
pooled_std = np.sqrt((real_std**2 + fake_std**2) / 2)
cohens_d = abs(real_mean - fake_mean) / pooled_std

print(f"\n{'='*50}")
print(f"SEPARATION ANALYSIS (PC1)")
print(f"{'='*50}")
print(f"Real:  mean = {real_mean:+.3f}, std = {real_std:.3f}")
print(f"Fake:  mean = {fake_mean:+.3f}, std = {fake_std:.3f}")
print(f"Cohen's d (effect size): {cohens_d:.3f}")
print(f"  - Small: d ≈ 0.2")
print(f"  - Medium: d ≈ 0.5")
print(f"  - Large: d ≥ 0.8")

In [ ]:
# Feature importance: which gradient features contribute most to separation?
feature_names = ['λ1 (max eigenval)', 'λ2 (min eigenval)', 'Trace', 'Determinant', 
                 'Anisotropy', 'Cov(Gx,Gy)', 'Mag Mean', 'Mag Std', '|Gx| Mean', '|Gy| Mean']

# PCA loadings (contribution of each feature to PCs)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 loadings
sorted_idx = np.argsort(np.abs(loadings[:, 0]))[::-1]
colors = ['green' if loadings[i, 0] > 0 else 'red' for i in sorted_idx]
axes[0].barh([feature_names[i] for i in sorted_idx], 
             [loadings[i, 0] for i in sorted_idx], color=colors, alpha=0.7)
axes[0].set_xlabel('PC1 Loading')
axes[0].set_title('Feature Contributions to PC1')
axes[0].axvline(x=0, color='black', linewidth=0.5)
axes[0].grid(True, alpha=0.3, axis='x')

# Individual feature distributions
# Show the most important feature
best_feature_idx = sorted_idx[0]
axes[1].hist(X[real_mask, best_feature_idx], bins=30, alpha=0.6, color='green', label='Real', density=True)
axes[1].hist(X[fake_mask, best_feature_idx], bins=30, alpha=0.6, color='red', label='Fake', density=True)
axes[1].set_xlabel(feature_names[best_feature_idx])
axes[1].set_ylabel('Density')
axes[1].set_title(f'Most Discriminative Feature: {feature_names[best_feature_idx]}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed feature comparison
print(f"\n{'='*70}")
print(f"FEATURE STATISTICS: REAL vs FAKE")
print(f"{'='*70}")
print(f"{'Feature':<20} {'Real (mean±std)':<22} {'Fake (mean±std)':<22} {'Separation':>10}")
print(f"{'-'*70}")

for i, name in enumerate(feature_names):
    r_mean, r_std = X[real_mask, i].mean(), X[real_mask, i].std()
    f_mean, f_std = X[fake_mask, i].mean(), X[fake_mask, i].std()
    pooled = np.sqrt((r_std**2 + f_std**2) / 2)
    d = abs(r_mean - f_mean) / (pooled + 1e-8)
    
    print(f"{name:<20} {r_mean:>8.4f} ± {r_std:<8.4f}  {f_mean:>8.4f} ± {f_std:<8.4f}  d={d:>6.3f}")

---
## 5. Visual Example: Real vs Fake Gradients

In [ ]:
def visualize_gradient_comparison(real_path, fake_path, img_size=256):
    """Side-by-side comparison of gradients for real and fake images."""
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    for row, (path, label) in enumerate([(real_path, 'REAL'), (fake_path, 'FAKE')]):
        # Load image
        img = Image.open(path).convert('RGB')
        img = img.resize((img_size, img_size))
        img_array = np.array(img) / 255.0
        
        # Process
        luminance = rgb_to_luminance(img_array)
        Gx, Gy = compute_gradients(luminance)
        magnitude = np.sqrt(Gx**2 + Gy**2)
        
        # Compute covariance
        gradient_matrix = np.stack([Gx.flatten(), Gy.flatten()], axis=1)
        cov = np.cov(gradient_matrix.T)
        eigenvalues = np.linalg.eigvalsh(cov)
        
        # Plot
        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f'{label}: Original')
        axes[row, 0].axis('off')
        
        axes[row, 1].imshow(luminance, cmap='gray')
        axes[row, 1].set_title(f'{label}: Luminance')
        axes[row, 1].axis('off')
        
        im = axes[row, 2].imshow(magnitude, cmap='hot')
        axes[row, 2].set_title(f'{label}: |∇L|\nmean={magnitude.mean():.4f}')
        axes[row, 2].axis('off')
        plt.colorbar(im, ax=axes[row, 2], fraction=0.046)
        
        # Gradient scatter (Gx vs Gy)
        subsample = np.random.choice(len(Gx.flatten()), size=2000, replace=False)
        axes[row, 3].scatter(Gx.flatten()[subsample], Gy.flatten()[subsample], 
                            alpha=0.3, s=5, c='blue')
        axes[row, 3].set_xlabel('Gx')
        axes[row, 3].set_ylabel('Gy')
        axes[row, 3].set_title(f'{label}: Gradient Distribution\nλ₁={eigenvalues[1]:.4f}, λ₂={eigenvalues[0]:.4f}')
        axes[row, 3].set_aspect('equal')
        axes[row, 3].grid(True, alpha=0.3)
        
        # Draw covariance ellipse
        from matplotlib.patches import Ellipse
        eigvecs = np.linalg.eigh(cov)[1]
        angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
        ellipse = Ellipse((0, 0), width=2*np.sqrt(eigenvalues[1])*3, height=2*np.sqrt(eigenvalues[0])*3,
                         angle=angle, fill=False, color='red', linewidth=2)
        axes[row, 3].add_patch(ellipse)
    
    plt.suptitle('Gradient Covariance Comparison: Real vs Fake', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Find a real and fake sample
real_path = test_paths[test_labels.index(0)]
fake_path = test_paths[test_labels.index(1)]

visualize_gradient_comparison(real_path, fake_path)

---
## 6. Quick Classifier Test

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

# Simple logistic regression on gradient covariance features
clf = LogisticRegression(max_iter=1000, random_state=42)

# Cross-validation
scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')
auroc_scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='roc_auc')

print(f"\n{'='*50}")
print(f"LOGISTIC REGRESSION (5-fold CV)")
print(f"{'='*50}")
print(f"Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"AUROC:    {auroc_scores.mean():.4f} ± {auroc_scores.std():.4f}")

# Fit on full data for report
clf.fit(X_scaled, y)
y_pred = clf.predict(X_scaled)

print(f"\nClassification Report (on full data):")
print(classification_report(y, y_pred, target_names=['Real', 'Fake']))